# Day 4: Spatial Intelligence
## Convolutional Neural Networks (CNNs) from Scratch
---
**Objective:** Understand how filters (kernels) extract features like edges and textures.

### Why CNNs?
In Day 3, we 'flattened' the image. Flattening destroys the spatial relationship (pixels next to each other are no longer 'neighbors'). CNNs fix this by using a sliding window.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

### 4.1 Manual Convolution (The Filter)
Let's see what a 'Vertical Edge' filter does to an image.

In [2]:
# Create a fake 5x5 image
image = torch.zeros(1, 1, 5, 5)
image[:, :, :, 2] = 1.0 # White line in the middle

# Vertical Edge Filter
filter = torch.tensor([[[[-1, 0, 1],
                         [-1, 0, 1],
                         [-1, 0, 1]]]], dtype=torch.float32)

output = F.conv2d(image, filter)
print("Original Image:\n", image.squeeze())
print("Filtered Output (Edges detected!):\n", output.squeeze())

Original Image:
 tensor([[0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.]])
Filtered Output (Edges detected!):
 tensor([[ 3.,  0., -3.],
        [ 3.,  0., -3.],
        [ 3.,  0., -3.]])


### 4.2 Explaining PyTorch CNN Layers
1. **nn.Conv2d(in_channels, out_channels, kernel_size)**: Slides a window over the image.
2. **nn.MaxPool2d(kernel_size)**: Shrinks the image by taking the maximum value in a window. This makes the model 'invariant' to small shifts in the image.

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: [1, 28, 28]
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1) # Output: [16, 28, 28]
        self.pool = nn.MaxPool2d(2, 2)                         # Output: [16, 14, 14]
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)# Output: [32, 14, 14]
        # Pool again: [32, 7, 7]
        
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # The Flattening Step: Critical transition from spatial to linear
        x = x.view(-1, 32 * 7 * 7)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

cnn_model = SimpleCNN()
print(cnn_model)

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=1568, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)


### 4.3 Inspecting Model Parameters
Let's see how many parameters a CNN uses compared to the MLP from Day 3.

In [4]:
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"Total CNN Parameters: {total_params:,}")
# Exercise: Compare this to the MLP parameters. You'll notice CNNs are much more efficient!

Total CNN Parameters: 105,866
